# ⚡ Rapid Lightning v3 — the head-aware edition

A clean, self-contained Colab notebook for the **best objective we found: head-aware fitness** — evolve the GP
ensemble members to be *signal generators* (each adds a new coordinate to the LR head's residual), not little
predictors. Train a tiny GP ensemble on tabular data → benchmark it against gradient-boosted trees → compile it
to a **~15 KB dependency-free, auditable C `.so`** with sub-µs CPU inference.

**What "best" means (honest).** Across our A/Bs, head-aware was the strongest single objective on every tier:
it beats `0/1-accuracy` and the `soft` margin on accuracy + AUC where there's headroom, and on **PR-AUC under
imbalance**, at **≤ the baseline's size**. On *continuous* data it even **Pareto-dominates XGBoost** (smaller
*and* better); on heavily *categorical* data it ties `soft` / trails XGBoost (it's a basis-match thing). The
two sibling ideas we tried — the `soft×head` hybrid and a stump dictionary — were honest **nulls** (head-aware
already subsumes them).

**What this is / isn't.** This is the **NumPy reference** at modest budget (like the research notebooks) — it
establishes the *behaviour* and the *artifact*. It is **not** the torch production engine (`rapid_lightning_v1/
v2`), and the bit-exact production-AOT port at ship budget (N≥25) is still the open #1 lead. So: honest
reference numbers, not a shipped benchmark.

**Colab:** pure NumPy + scikit-learn + matplotlib (all preinstalled); `xgboost` optional. Run top to bottom;
tweak the **CONFIG** cell.

## 1 · The engine — GP substrate, head-aware fitness, LR head, datasets, auditable-C compiler
(Validated code, transcribed verbatim. `load_dataset` gives `synthetic` / `adult` / `bank`; bring your own in §5.)

In [1]:
"""
Head-aware fitness — decisive cheap check.

Question #1 (make-or-break for this project): do head-aware members grow the
trees? total node count == what the AOT .so size/latency scales with.
Question #2: does the head-aware arm even produce a competitive LR-head model?

Three arms share ONE GP engine (production tier-3 ops incl `lt`, max_depth=8,
parsimony=1e-4, islands+migration+elitism). Only the member objective changes:
  acc   : 0/1 accuracy   + AdaBoost(majority-vote) sequencing   (production default)
  soft  : signed standardized class-balanced margin + AdaBoost   (shipped objective)
  head  : residual-of-the-LR-head / orthogonal-to-pool / corr    (replaces AdaBoost)

Then compile each arm's full stack (GP trees -> standardize -> LR -> sigmoid) to C
via the XGForge pipeline and measure real .so KB + ns/row.
"""
import random, time, os, tempfile, subprocess, ctypes, sys
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

SUBSAMPLE=2500
def load_dataset(dataset, seed, sub=SUBSAMPLE):
    if dataset=="synthetic":
        X,y=make_classification(n_samples=5500,n_features=8,n_informative=6,n_redundant=1,
                                n_clusters_per_class=2,class_sep=0.9,flip_y=0.03,random_state=seed)
        X=X.astype(F32); y=y.astype(np.int64)
        return train_test_split(X,y,test_size=4000,random_state=seed,stratify=y)
    name,ver=("adult",2) if dataset=="adult" else ("bank-marketing",1)
    d=fetch_openml(name,version=ver,as_frame=True,parser="auto")
    Xdf=d.data.copy(); ys=d.target
    if dataset=="adult":
        y=(ys.astype(str).str.replace(".","",regex=False)==">50K").astype(int).values
    else:
        y=ys.astype(str).isin(["2","yes"]).astype(int).values
    Xdf=Xdf.replace("?",np.nan)
    cat=Xdf.select_dtypes(include=["object","category"]).columns.tolist()
    num=Xdf.select_dtypes(include=["number"]).columns.tolist()
    for c in cat: Xdf[c]=Xdf[c].fillna(Xdf[c].mode().iloc[0])
    for c in num: Xdf[c]=Xdf[c].fillna(Xdf[c].median())
    pre=ColumnTransformer([("num",StandardScaler(),num),
        ("cat",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),cat)])
    Xtr_df,Xte_df,ytr,yte=train_test_split(Xdf,y,test_size=0.2,random_state=seed,stratify=y)
    pre.fit(Xtr_df); s2=StandardScaler().fit(pre.transform(Xtr_df))
    Xtr=s2.transform(pre.transform(Xtr_df)).astype(F32)
    Xte=s2.transform(pre.transform(Xte_df)).astype(F32)
    if sub<len(Xtr):
        Xtr,_,ytr,_=train_test_split(Xtr,ytr,train_size=sub,random_state=seed,stratify=ytr)
    if len(Xte)>3000:
        Xte,_,yte,_=train_test_split(Xte,yte,train_size=3000,random_state=seed,stratify=yte)
    return Xtr,Xte,ytr,yte

F32 = np.float32

# -------- budget (reduced for a fast directional read; production is 25/80) --------
NUM_MEMBERS = int(os.environ.get("HA_MEMBERS", 8))
GENS        = int(os.environ.get("HA_GENS", 20))
POP         = int(os.environ.get("HA_POP", 160))
SEED        = int(os.environ.get("HA_SEED", 1))

# ======================= GP engine (production tier-3) =======================
OPS = {'add':2,'mul':2,'sub':2,'max':2,'min':2,'neg':1,'abs':1,'clamp':1,'lt':2}
ACTIVE = ['add','mul','sub','max','min','neg','abs','clamp','lt']
NF = None  # set at runtime

class GPNode:
    __slots__ = ['ntype','value','children']
    def __init__(self, ntype, value, children=None):
        self.ntype=ntype; self.value=value; self.children=children or []
    def evaluate(self, X):
        if self.ntype=='input': return X[:, self.value % X.shape[1]]
        if self.ntype=='const': return np.full(X.shape[0], self.value, dtype=F32)
        a=[c.evaluate(X) for c in self.children]; op=self.value
        if   op=='add':   r=a[0]+a[1]
        elif op=='mul':   r=a[0]*a[1]
        elif op=='sub':   r=a[0]-a[1]
        elif op=='max':   r=np.maximum(a[0],a[1])
        elif op=='min':   r=np.minimum(a[0],a[1])
        elif op=='neg':   r=-a[0]
        elif op=='abs':   r=np.abs(a[0])
        elif op=='clamp': r=np.clip(a[0],0.0,1.0)
        elif op=='lt':    r=(a[0]<a[1]).astype(F32)
        else:             r=a[0]
        return np.clip(r,-1e6,1e6).astype(F32)
    def size(self):  return 1+sum(c.size() for c in self.children)
    def depth(self): return 1 if not self.children else 1+max(c.depth() for c in self.children)
    def clone(self): return GPNode(self.ntype,self.value,[c.clone() for c in self.children])
    def all_nodes(self):
        out=[self]
        for c in self.children: out.extend(c.all_nodes())
        return out

class GPIndividual:
    __slots__ = ['trees','fitness','train_acc']
    def __init__(self, trees, fitness=0.0, train_acc=0.0):
        self.trees=trees; self.fitness=fitness; self.train_acc=train_acc
    def get_scores(self, X): return np.stack([t.evaluate(X) for t in self.trees], axis=1)
    def classify(self, X):   return self.get_scores(X).argmax(axis=1)
    def total_size(self):    return sum(t.size() for t in self.trees)
    def max_depth(self):     return max(t.depth() for t in self.trees)
    def clone(self): return GPIndividual([t.clone() for t in self.trees], self.fitness, self.train_acc)

def random_tree(md_, ins, d=0):
    if d>=md_ or (d>1 and random.random()<0.3):
        if random.random()<0.7: return GPNode('input', random.randint(0,ins-1))
        return GPNode('const', random.uniform(-2,2))
    op=random.choice(ACTIVE)
    return GPNode('op', op, [random_tree(md_,ins,d+1) for _ in range(OPS[op])])

def random_individual(nc, md_, ins): return GPIndividual([random_tree(md_,ins) for _ in range(nc)])

def mutate(ind, md_, ins):
    m=ind.clone(); ti=random.randint(0,len(m.trees)-1); nodes=m.trees[ti].all_nodes()
    if len(nodes)<=1 or random.random()<0.3:
        m.trees[ti]=random_tree(md_,ins)
    else:
        t=random.choice(nodes)
        if   t.ntype=='const': t.value+=random.gauss(0,0.5)
        elif t.ntype=='input': t.value=random.randint(0,ins-1)
        else:
            n=random_tree(min(3,md_-1),ins); t.ntype,t.value,t.children=n.ntype,n.value,n.children
    return m

def xover(p1, p2, md_, ins):
    ch=p1.clone(); ti=random.randint(0,len(ch.trees)-1)
    n1,n2=ch.trees[ti].all_nodes(), p2.trees[ti].all_nodes()
    if len(n1)>1 and len(n2)>1:
        tgt=random.choice(n1); don=random.choice(n2).clone()
        tgt.ntype,tgt.value,tgt.children=don.ntype,don.value,don.children
    if ch.trees[ti].depth()>md_+2: ch.trees[ti]=random_tree(md_,ins)
    return ch

def tsel(pop, sz=7):
    best=None
    for _ in range(sz):
        c=random.choice(pop)
        if best is None or c.fitness>best.fitness: best=c
    return best

class Cfg:
    pop_size=POP; num_islands=4; max_depth=8; gens=GENS
    migrate_every=12; migrate_count=3; tournament_size=7
    crossover_rate=0.85; mutation_rate=0.20; parsimony=0.0001
    elite_pct=0.05; fresh_pct=0.03; boost_weight=3.0; boost_after=3
    num_members=NUM_MEMBERS; seed=SEED

def evolve_member(X, y, cfg, eval_fn, weights=None, member_num=0):
    ins=X.shape[1]; nc=int(y.max())+1
    random.seed(cfg.seed + member_num*1000)
    ppi=cfg.pop_size//cfg.num_islands
    islands=[[random_individual(nc,cfg.max_depth,ins) for _ in range(ppi)] for _ in range(cfg.num_islands)]
    best, best_fit = None, -np.inf
    for gen in range(1, cfg.gens+1):
        for i in range(len(islands)):
            pop=islands[i]; eval_fn(pop, X, y, cfg.parsimony, weights)
            pop.sort(key=lambda x:x.fitness, reverse=True)
            if pop[0].fitness>best_fit: best_fit=pop[0].fitness; best=pop[0].clone()
            new=[]; ec=max(2,int(len(pop)*cfg.elite_pct))
            for j in range(ec): new.append(pop[j].clone())
            while len(new)<len(pop):
                p1,p2=tsel(pop,cfg.tournament_size),tsel(pop,cfg.tournament_size)
                ch=xover(p1,p2,cfg.max_depth,ins) if random.random()<cfg.crossover_rate else p1.clone()
                if random.random()<cfg.mutation_rate: ch=mutate(ch,cfg.max_depth,ins)
                new.append(ch)
            for j in range(int(len(pop)*cfg.fresh_pct)):
                ri=len(pop)-1-j
                if ri>=ec: new[ri]=random_individual(nc,cfg.max_depth,ins)
            islands[i]=new
        if gen % cfg.migrate_every == 0:
            ni=len(islands)
            for i in range(ni):
                src=sorted(islands[i], key=lambda x:x.fitness, reverse=True)
                tgt=sorted(islands[(i+1)%ni], key=lambda x:x.fitness)
                for mm in range(min(cfg.migrate_count,len(src),len(tgt))): tgt[mm]=src[mm].clone()
                islands[(i+1)%ni]=tgt
    return best

def majority_vote(members, X, nc):
    P=np.stack([m.classify(X) for m in members], axis=0)
    return np.array([np.bincount(P[:,j], minlength=nc).argmax() for j in range(X.shape[0])])

# ======================= the three fitness functions =======================
def make_acc_eval(y, weights):
    def eval_pop(pop, X, y2, pars, _w=None):
        for ind in pop:
            try:
                correct=(ind.classify(X)==y)
                ind.train_acc=float((correct*weights).sum()/weights.sum()) if weights is not None else float(correct.mean())
                ind.fitness=ind.train_acc - ind.total_size()*pars
            except Exception:
                ind.fitness=0.0; ind.train_acc=0.0
    return eval_pop

def make_soft_eval(y, weights):
    # production soft: signed standardized class-balanced margin (binary)
    yf=y.astype(np.float64); pr=float(yf.mean()); sgn=2*yf-1
    cb=np.where(y==1, 0.5/max(pr,1e-6), 0.5/max(1-pr,1e-6))
    if weights is not None: cb=cb*weights
    w=cb/cb.sum()
    def eval_pop(pop, X, y2, pars, _w=None):
        for ind in pop:
            try:
                s=np.nan_to_num(ind.get_scores(X).astype(np.float64), nan=0.0, posinf=1e6, neginf=-1e6)
                d=s[:,1]-s[:,0]; d=(d-d.mean())/(d.std()+1e-6)
                f=float((w*sgn*d).sum())
                ind.fitness=f - ind.total_size()*pars
                ind.train_acc=float((s.argmax(1)==y2).mean())
            except Exception:
                ind.fitness=-1e9; ind.train_acc=0.0
    return eval_pop

def stack(members, X):
    return np.concatenate([m.get_scores(X) for m in members], axis=1).astype(np.float64)

def pool_setup(members, X, y, nc):
    N=X.shape[0]; Y1=np.eye(nc,dtype=np.float64)[y]; cols=[np.ones((N,1))]
    if members:
        Phi=stack(members,X); Phs=StandardScaler().fit_transform(Phi); cols.append(Phs)
        lr=LogisticRegression(max_iter=2000,C=1.0,solver='lbfgs').fit(Phs,y)
        P=np.zeros((N,nc))
        for j,c in enumerate(lr.classes_): P[:,c]=lr.predict_proba(Phs)[:,j]
    else:
        P=np.tile(np.bincount(y,minlength=nc)/N,(N,1))
    B=np.concatenate(cols,axis=1)
    Bproj=B @ np.linalg.inv(B.T@B + 1e-3*np.eye(B.shape[1]))
    return (Y1-P), Bproj, B

def make_headaware_eval(R, Bproj, B):
    Rc=R-R.mean(0); Rn=np.sqrt((Rc**2).sum(0))+1e-12
    def eval_pop(pop, X, y, pars, _w=None):
        for ind in pop:
            try:
                S=np.nan_to_num(ind.get_scores(X).astype(np.float64), nan=0.0, posinf=1e6, neginf=-1e6)
                S_perp=S - Bproj @ (B.T @ S)
                Sc=S_perp-S_perp.mean(0); Sn=np.sqrt((Sc**2).sum(0))+1e-12
                corr=(Sc*Rc).sum(0)/(Sn*Rn)
                ind.fitness=float((corr**2).sum()) - ind.total_size()*pars
                ind.train_acc=ind.fitness
            except Exception:
                ind.fitness=-1e9; ind.train_acc=0.0
    return eval_pop

# --- ARM 4: hybrid (soft x head) — head-aware with a CLASS-BALANCED, weighted residual ---
def pool_setup_balanced(members, X, y, nc):
    # like pool_setup but: (1) inverse-prevalence sample weights w; (2) residual from a
    # class_weight='balanced' LR; (3) the orthogonalization uses the w-inner-product.
    N=X.shape[0]; Y1=np.eye(nc,dtype=np.float64)[y]; cols=[np.ones((N,1))]
    if nc==2:
        pr=float(y.mean()); cb=np.where(y==1,0.5/max(pr,1e-6),0.5/max(1-pr,1e-6)); w=cb/cb.sum()
    else:
        w=np.full(N,1.0/N)
    if members:
        Phi=stack(members,X); Phs=StandardScaler().fit_transform(Phi); cols.append(Phs)
        lr=LogisticRegression(max_iter=2000,C=1.0,solver='lbfgs',class_weight='balanced').fit(Phs,y)
        P=np.zeros((N,nc))
        for j,c in enumerate(lr.classes_): P[:,c]=lr.predict_proba(Phs)[:,j]
    else:
        P=np.tile(np.bincount(y,minlength=nc)/N,(N,1))
    B=np.concatenate(cols,axis=1)
    BtWB=B.T @ (w[:,None]*B) + 1e-3*np.eye(B.shape[1])
    Bproj_w=B @ np.linalg.inv(BtWB)                      # weighted projector
    return (Y1-P), Bproj_w, B, w

def make_hybrid_eval(R, Bproj_w, B, w):
    wv=w[:,None]
    Rm=(wv*R).sum(0); Rc=R-Rm; Rn=np.sqrt((wv*Rc*Rc).sum(0))+1e-12   # weighted moments
    def eval_pop(pop, X, y, pars, _w=None):
        for ind in pop:
            try:
                S=np.nan_to_num(ind.get_scores(X).astype(np.float64), nan=0.0, posinf=1e6, neginf=-1e6)
                S_perp=S - Bproj_w @ (B.T @ (wv*S))      # w-orthogonal to the pool
                Sm=(wv*S_perp).sum(0); Sc=S_perp-Sm; Sn=np.sqrt((wv*Sc*Sc).sum(0))+1e-12
                corr=(wv*Sc*Rc).sum(0)/(Sn*Rn)           # weighted correlation, direction-only
                ind.fitness=float((corr**2).sum()) - ind.total_size()*pars
                ind.train_acc=ind.fitness
            except Exception:
                ind.fitness=-1e9; ind.train_acc=0.0
    return eval_pop

In [2]:
# ======================= ensemble sequencing =======================
def evolve_ensemble_boost(X, y, cfg, make_eval):
    nc=int(y.max())+1; members=[]
    for m in range(1, cfg.num_members+1):
        weights=None
        if m>cfg.boost_after and len(members)>=cfg.boost_after:
            wrong=(majority_vote(members,X,nc)!=y)
            w=np.ones(X.shape[0],dtype=np.float64); w[wrong]=cfg.boost_weight; weights=w/w.mean()
        members.append(evolve_member(X,y,cfg,make_eval(y,weights),weights=weights,member_num=m))
    return members

def evolve_ensemble_headaware(X, y, cfg):
    nc=int(y.max())+1; members=[]
    for m in range(1, cfg.num_members+1):
        R,Bproj,B=pool_setup(members,X,y,nc)
        members.append(evolve_member(X,y,cfg,make_headaware_eval(R,Bproj,B),weights=None,member_num=m))
    return members

def evolve_ensemble_hybrid(X, y, cfg):
    nc=int(y.max())+1; members=[]
    for m in range(1, cfg.num_members+1):
        R,Bproj_w,B,w=pool_setup_balanced(members,X,y,nc)
        members.append(evolve_member(X,y,cfg,make_hybrid_eval(R,Bproj_w,B,w),weights=None,member_num=m))
    return members

# ======================= head + metrics =======================
def lr_head(members, Xtr, ytr, Xte, yte):
    Ztr,Zte=stack(members,Xtr),stack(members,Xte)
    sc=StandardScaler().fit(Ztr)
    lr=LogisticRegression(max_iter=2000,C=1.0,solver='lbfgs').fit(sc.transform(Ztr),ytr)
    p=lr.predict_proba(sc.transform(Zte))[:,1]
    acc=float(lr.score(sc.transform(Zte),yte))
    auc=float(roc_auc_score(yte,p)); pr=float(average_precision_score(yte,p))
    return acc, auc, pr, sc, lr

def mlp_head(members, Xtr, ytr, Xte, yte):
    # directional probe: does a nonlinear head exploit head-aware's complementary features?
    Ztr,Zte=stack(members,Xtr),stack(members,Xte)
    sc=StandardScaler().fit(Ztr)
    mlp=MLPClassifier(hidden_layer_sizes=(32,),alpha=1e-3,max_iter=800,random_state=0)\
        .fit(sc.transform(Ztr),ytr)
    p=mlp.predict_proba(sc.transform(Zte))[:,1]
    return float(mlp.score(sc.transform(Zte),yte)), float(roc_auc_score(yte,p))

def member_diversity(members, X):
    P=np.stack([m.classify(X) for m in members],axis=0); M=P.shape[0]
    tot,cnt=0.0,0
    for i in range(M):
        for j in range(i+1,M):
            tot+=float((P[i]!=P[j]).mean()); cnt+=1
    return tot/max(cnt,1)

# ======================= GP-stack -> C (XGForge pipeline) =======================
def fmt_f(v):
    v=float(v); s=f"{v:.9g}"
    if '.' not in s and 'e' not in s and 'E' not in s and 'n' not in s and 'i' not in s: s=f"{v:.1f}"
    return '('+s+'f)'   # parenthesize so negative literals never form `--` / `-+`

def emit_node(n):
    if n.ntype=='input': return f"X[{n.value % NF}]"
    if n.ntype=='const': return fmt_f(n.value)
    a=[emit_node(c) for c in n.children]; op=n.value
    if   op=='add':   return f"({a[0]}+{a[1]})"
    elif op=='sub':   return f"({a[0]}-{a[1]})"
    elif op=='mul':   return f"({a[0]}*{a[1]})"
    elif op=='max':   return f"fmaxf({a[0]},{a[1]})"
    elif op=='min':   return f"fminf({a[0]},{a[1]})"
    elif op=='neg':   return f"(-({a[0]}))"
    elif op=='abs':   return f"fabsf({a[0]})"
    elif op=='clamp': return f"fminf(fmaxf({a[0]},0.0f),1.0f)"
    elif op=='lt':    return f"(({a[0]}<{a[1]})?1.0f:0.0f)"
    return a[0]

TIMING_C = '''
double predict_proba_time_ns(const float* X, int nw, int ni){
    volatile float sink=0; for(int i=0;i<nw;++i) sink+=predict_proba_one(X);
    struct timespec t0,t1; clock_gettime(CLOCK_MONOTONIC,&t0);
    for(int i=0;i<ni;++i) sink+=predict_proba_one(X);
    clock_gettime(CLOCK_MONOTONIC,&t1); (void)sink;
    double dt=(double)(t1.tv_sec-t0.tv_sec)*1e9+(double)(t1.tv_nsec-t0.tv_nsec);
    return dt/(double)ni;
}'''

def forge_gp_stack(members, scaler, lr):
    lines=['#include <math.h>','#include <time.h>','',
           'float predict_proba_one(const float* X){']
    k=0; terms=[]
    mean=scaler.mean_; scale=scaler.scale_
    coef=lr.coef_[0]; b=float(lr.intercept_[0])
    for mem in members:
        for t in mem.trees:
            lines.append(f"    float f{k} = (float)({emit_node(t)});")
            c=coef[k]/scale[k]
            terms.append(f"({fmt_f(c)}*(f{k}-{fmt_f(mean[k])}))")
            k+=1
    lines.append(f"    float s = {fmt_f(b)} + " + " + ".join(terms) + ";")
    lines.append("    return 1.0f/(1.0f+expf(-s));")
    lines.append("}")
    lines.append(TIMING_C)
    src="\n".join(lines)
    d=tempfile.mkdtemp(prefix="hagp_"); sp=os.path.join(d,"p.c"); op=os.path.join(d,"p.so")
    open(sp,"w").write(src)
    subprocess.run(['gcc','-O3','-march=native','-ffast-math','-shared','-fPIC','-o',op,sp],check=True)
    lib=ctypes.CDLL(op)
    lib.predict_proba_one.restype=ctypes.c_float
    lib.predict_proba_one.argtypes=[ctypes.POINTER(ctypes.c_float)]
    lib.predict_proba_time_ns.restype=ctypes.c_double
    lib.predict_proba_time_ns.argtypes=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_int]
    return lib, os.path.getsize(op)/1024, op

## 2 · CONFIG — train a head-aware ensemble

In [3]:
# ----------------------------- CONFIG -----------------------------
DATASET     = "synthetic"     # "synthetic" | "adult" | "bank"  (or use load_csv() in §5)
NUM_MEMBERS = 10              # ensemble size (small, like a fast runtime)
GENS        = 20              # GP generations / member  (bump to 30-40 for stronger numbers)
POP         = 160             # population / member
SEED        = 1
# ------------------------------------------------------------------
import numpy as np
Xtr, Xte, ytr, yte = load_dataset(DATASET, SEED)
NF = Xtr.shape[1]            # global used by the C emitter
cfg = Cfg; cfg.seed = SEED; cfg.num_members = NUM_MEMBERS; cfg.gens = GENS; cfg.pop_size = POP
print(f"[{DATASET}] train={len(Xtr)} test={len(Xte)} features={NF} pos%={ytr.mean():.1%}  "
      f"-> evolving {NUM_MEMBERS} head-aware members (gens={GENS})…")
import time; t0=time.time()
members = evolve_ensemble_headaware(Xtr, ytr, cfg)
acc, auc, pr, scaler, lr = lr_head(members, Xtr, ytr, Xte, yte)
nodes = sum(m.total_size() for m in members)
mem_acc = float(np.mean([(m.classify(Xte)==yte).mean() for m in members]))
div = member_diversity(members, Xte)
print(f"evolved in {time.time()-t0:.0f}s | nodes={nodes}")
print(f"  test  acc={acc:.4f}  ROC-AUC={auc:.4f}  PR-AUC={pr:.4f}")
print(f"  mechanism: mean member acc={mem_acc:.3f} (toward chance), diversity={div:.3f} (high) "
      f"— members are signal generators, the head classifies")

[synthetic] train=1500 test=4000 features=8 pos%=50.3%  -> evolving 10 head-aware members (gens=20)…


evolved in 17s | nodes=385
  test  acc=0.8860  ROC-AUC=0.9499  PR-AUC=0.9548
  mechanism: mean member acc=0.573 (toward chance), diversity=0.487 (high) — members are signal generators, the head classifies


## 3 · Benchmark vs gradient-boosted trees (the yardstick)
Two comparisons: **full** GBDT/XGBoost (the accuracy *ceiling*, but 5–10× the size), and a **size-matched** XGBoost (≈ the GP's node count) — the honest iso-size comparison, where head-aware tends to win on continuous data.

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
def m3(p): return float(((p>0.5)==yte).mean()), float(roc_auc_score(yte,p)), float(average_precision_score(yte,p))
rows = [("Rapid Lightning v3 (head-aware)", acc, auc, pr, nodes)]
gb = HistGradientBoostingClassifier(max_iter=300, max_depth=6, learning_rate=0.08,
                                    l2_regularization=1.0, random_state=SEED).fit(Xtr, ytr)
rows.append(("HistGradientBoosting (full)", *m3(gb.predict_proba(Xte)[:,1]), None))
try:
    import xgboost as xgb
    xf = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08, eval_metric="logloss",
                           random_state=SEED, verbosity=0).fit(Xtr, ytr)
    rows.append(("XGBoost (full)", *m3(xf.predict_proba(Xte)[:,1]), len(xf.get_booster().trees_to_dataframe())))
    best = None                                    # size-match XGBoost to the GP's node count
    for k in [2,3,5,8,13,21,34,55]:
        xm = xgb.XGBClassifier(n_estimators=k, max_depth=4, learning_rate=0.3, eval_metric="logloss",
                               random_state=SEED, verbosity=0).fit(Xtr, ytr)
        xn = len(xm.get_booster().trees_to_dataframe())
        if best is None or abs(xn-nodes) < abs(best[1]-nodes): best = (xm, xn)
    rows.append((f"XGBoost (size-matched ≈{best[1]}n)", *m3(best[0].predict_proba(Xte)[:,1]), best[1]))
except Exception:
    print("(xgboost not available — showing HistGB only)")
print(f"{'model':<34}{'acc':>8}{'ROC-AUC':>9}{'PR-AUC':>9}{'nodes':>8}")
for nm,a,u,p,n in rows: print(f"{nm:<34}{a:>8.4f}{u:>9.4f}{p:>9.4f}{(str(n) if n else '—'):>8}")
print("\nKey: vs the SIZE-MATCHED XGBoost (same node budget = same .so size/latency class), head-aware should")
print("hold its own or win on continuous data — that's the 'smaller AND better' frontier point. Full GBDT is")
print("the ceiling (5-10x bigger). See research/headaware/headaware_frontier_v1 for the full sweep.")

model                                  acc  ROC-AUC   PR-AUC   nodes
Rapid Lightning v3 (head-aware)     0.8860   0.9499   0.9548     385
HistGradientBoosting (full)         0.9185   0.9657   0.9656       —
XGBoost (full)                      0.9197   0.9650   0.9656   10614
XGBoost (size-matched ≈317n)        0.8748   0.9437   0.9502     317

Key: vs the SIZE-MATCHED XGBoost (same node budget = same .so size/latency class), head-aware should
hold its own or win on continuous data — that's the 'smaller AND better' frontier point. Full GBDT is
the ceiling (5-10x bigger). See research/headaware/headaware_frontier_v1 for the full sweep.


## 4 · Compile to the tiny auditable C artifact (the ship object)
`forge_gp_stack` lowers the ensemble → straight-line C (members → standardize → logistic head → sigmoid), compiles with `gcc -O3 -march=native -ffast-math`, and C-times single-row inference. The `.so` input is the model's feature vector (apply your dataset's preprocessing first).

In [5]:
import os, ctypes
NF = Xtr.shape[1]
lib, kb, so_path = forge_gp_stack(members, scaler, lr)
x0 = np.ascontiguousarray(Xte[0], dtype=np.float32)
ns = lib.predict_proba_time_ns(x0.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), 5000, 1_000_000)
# C <-> Python agreement (rank/AUC)
cprob = np.array([lib.predict_proba_one(np.ascontiguousarray(Xte[i],dtype=np.float32).ctypes.data_as(
        ctypes.POINTER(ctypes.c_float))) for i in range(min(3000,len(Xte)))])
cauc = float(roc_auc_score(yte[:len(cprob)], cprob))
LINE = 1_488_095   # 1 GbE, 64B frames
print(f".so = {kb:.1f} KB   {ns:.1f} ns/row   = {1e9/ns/1e6:.1f}M inferences/sec/core   "
      f"({1e9/ns/LINE:.0f}× a 1 GbE link)")
print(f"C↔Python AUC: {cauc:.4f} vs {auc:.4f}  (fp32 reference emitter; ship via the bit-exact AOT path)")
print("\n--- ids_model.c (head) ---")
print("".join(open(os.path.join(os.path.dirname(so_path), "p.c")).readlines()[:8]), "  … (one f-block / tree, then the logistic head)")
# On Colab, download the artifact:
try:
    from google.colab import files; import shutil
    shutil.copy(so_path, "rapid_lightning_v3.so"); shutil.copy(os.path.join(os.path.dirname(so_path),"p.c"), "rapid_lightning_v3.c")
    print("\nsaved rapid_lightning_v3.so / .c — uncomment to download:")
    print("# files.download('rapid_lightning_v3.so'); files.download('rapid_lightning_v3.c')")
except Exception:
    pass

.so = 15.3 KB   49.8 ns/row   = 20.1M inferences/sec/core   (13× a 1 GbE link)
C↔Python AUC: 0.9481 vs 0.9499  (fp32 reference emitter; ship via the bit-exact AOT path)

--- ids_model.c (head) ---
#include <math.h>
#include <time.h>

float predict_proba_one(const float* X){
    float f0 = (float)((-(((X[1]+fminf(fmaxf(X[0],0.0f),1.0f))+fminf(fmaxf(fminf(fmaxf(X[1],0.0f),1.0f),0.0f),1.0f)))));
    float f1 = (float)((X[0]+fminf((-((((-(X[1]))-X[0])+fmaxf((((-(X[1]))-X[0])+fmaxf((X[3]-(-1.40290373f)),((X[4]<X[6])?1.0f:0.0f))),((X[4]<X[6])?1.0f:0.0f))))),fmaxf((X[3]-(-1.40290373f)),((X[4]<X[6])?1.0f:0.0f)))));
    float f2 = (float)(fmaxf(fminf(X[5],X[5]),fabsf(fminf(fabsf((-(X[7]))),(-(X[7]))))));
    float f3 = (float)(fmaxf(fminf((X[7]-X[5]),X[7]),(-(fminf((-((fminf(fmaxf(X[7],0.0f),1.0f)-(((X[2]*(-0.746064527f))+X[5])*(((((-1.15864666f)<(X[7]+X[1]))?1.0f:0.0f)<(X[7]+X[1]))?1.0f:0.0f))))),fminf((X[7]-X[5]),X[7]))))));
   … (one f-block / tree, then the logistic head)


## 5 · Bring your own data (CSV)
Point at a CSV + name the target column. Numeric columns are standardized, categoricals ordinal-encoded (binary targets; >2 classes need the multiclass path). Then re-run §2–4.

In [6]:
def load_csv(path, target, test_size=0.3, seed=1, subsample=8000):
    import pandas as pd
    from sklearn.preprocessing import StandardScaler, OrdinalEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.model_selection import train_test_split
    df = pd.read_csv(path).dropna(axis=1, how="all")
    y = df[target]
    y = (y.astype("category").cat.codes.values if y.dtype == object or str(y.dtype) == "category"
         else (y.values != y.mode()[0]).astype(int) if y.nunique() > 2 else (y.values == y.unique()[-1]).astype(int))
    X = df.drop(columns=[target])
    cat = X.select_dtypes(include=["object","category"]).columns.tolist()
    num = [c for c in X.columns if c not in cat]
    for c in cat: X[c] = X[c].fillna("?" )
    for c in num: X[c] = X[c].fillna(X[c].median())
    pre = ColumnTransformer([("n", StandardScaler(), num),
        ("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)])
    Xt, Xv, yt, yv = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y if len(set(y))>1 else None)
    s2 = StandardScaler().fit(pre.fit_transform(Xt))
    Xt2 = s2.transform(pre.transform(Xt)).astype(np.float32); Xv2 = s2.transform(pre.transform(Xv)).astype(np.float32)
    if subsample and len(Xt2) > subsample:
        Xt2, _, yt, _ = train_test_split(Xt2, yt, train_size=subsample, random_state=seed)
    print(f"loaded {path}: train={len(Xt2)} test={len(Xv2)} features={Xt2.shape[1]} pos%={yt.mean():.1%}")
    return Xt2, Xv2, yt, yv

# Example — then re-run §2 (skip its load_dataset line) and §3-4:
# Xtr, Xte, ytr, yte = load_csv("/content/mydata.csv", target="label"); NF = Xtr.shape[1]
print("load_csv() ready — see the commented example above.")

load_csv() ready — see the commented example above.


## 6 · What this is / honest caveats / next

- **Head-aware is the strongest objective we found** — and it's what the IDS Pi POC runs on (`ids/pi_poc/`). On
  continuous tabular data it can beat gradient-boosted trees on the **accuracy↔size frontier** (smaller *and*
  better); on heavily categorical data it ties `soft` / trails XGBoost.
- **This is the NumPy reference**, not the torch production engine (`engine/rapid_lightning_v1/v2`) — and
  reduced budget. The bit-exact **production-AOT port at ship budget (N≥25)** is the open #1 lead in
  `NEXT_SESSION_HANDOFF.md`; that's what turns these honest reference numbers into a shipped benchmark.
- **The `.so` input is the model's feature vector** — replicate your dataset's preprocessing before calling it
  (the C folds the GP→stack→LR→sigmoid, not the input encoding). The fp32 reference emitter ≈ exact on AUC;
  ship via the bit-exact `lightning_aot` path for production.
- **Binary targets** here; multiclass needs the one-vs-rest head-aware path (used in the production engine).
- **Try:** bump `GENS`/`NUM_MEMBERS`; switch `DATASET` (`synthetic` shows the XGBoost-beating frontier best);
  bring your own CSV (§5); compare objectives by swapping `evolve_ensemble_headaware` for
  `evolve_ensemble_boost(Xtr, ytr, cfg, make_acc_eval)` / `make_soft_eval`.